# 可視化

In [1]:
import vtk
import pyvista as pv
import pyvistaqt as pvqt
from helper.pyvista import plotter_to_iframe

## Slice

三次元データを表示する比較的簡単な方法のひとつは、スライス（切断面）を用いることです。スライスによって通過するデータを可視化することで、三次元データの可視化の問題を二次元の可視化に変換できます。スライスの位置や方向をインタラクティブに変更することで、ユーザーは三次元データを直感的に観察することができます。

次のコードは`MultiBlockPlot3DReader()`を使ってデータファイルからデータを読み込みます。`MultiBlockPLOT3DReader` は、PLOT3D形式のファイルを読み込むためのリーダーです。PLOT3Dは、流体力学データの可視化に用いられる形式です。このプログラムでは、`add_function()` メソッドを使って、読み込むスカラー配列とベクトル配列を追加します。各パラメータの詳細については、VTKの公式ドキュメントを参照してください。`MultiBlockPLOT3DReader` オブジェクトの出力は、`StructuredGrid` 型のデータセットになります。

In [2]:
reader = pv.reader.MultiBlockPlot3DReader('data/combxyz.bin')
reader.add_q_files('data/combq.bin')
reader.add_function(100)
reader.add_function(153)
reader.add_function(200)
blocks = reader.read()
grid = blocks[0]
grid

StructuredGrid (0x2163ee0b4c0)
  N Cells:      43008
  N Points:     47025
  X Bounds:     0.000e+00, 1.651e+01
  Y Bounds:     -5.662e+00, 5.662e+00
  Z Bounds:     2.333e+01, 3.619e+01
  Dimensions:   57, 33, 25
  N Arrays:     6

上記の各属性から、データセット `grid` は形状が `(57, 33, 25)` のグリッドであることがわかります。このグリッドは `StructuredGrid` オブジェクトであるため、各グリッド点の座標は `points` 属性に保存されています。

このグリッドのセル（セル単位のデータ）は存在しませんが、各点には3種類のデータが対応しており、それぞれの名前は `"Density"`、`"Momentum"`、`"StagnationEnergy"`、`"VelocityMagnitude"`と`"Velocity"`です。

- `"Density"` の配列は、`point_data` の `active_scalars` 属性を通じて取得でき、スカラー配列です。
- `"Momentum"` の配列は、`point_data` の `active_vectors` 属性から取得でき、3次元のベクトル配列です。

スライス処理は `active_scalars` データを元に実行されるため、スライスで表示されるのはスライス面上の各点の密度（Density）になります。

In [3]:
print(grid.point_data.active_scalars_name, grid.point_data.active_vectors_name)

Density Momentum


`pyvista` の `slice` 関連のフィルタは、3D グリッドデータから2Dの断面（スライス）を抽出するための手段です。以下のテーブルは、各メソッドの用途をまとめます。

| メソッド名               | 特徴                                           |
|--------------------------|------------------------------------------------|
| `slice()`                | 任意の法線による1枚のスライス                |
| `slice_along_axis()`     | 軸に沿って等間隔の複数スライス               |
| `slice_along_line()`     | 線に沿ったスライス                             |
| `slice_implicit()`       | 球・平面などの形状でスライス（非線形もOK）    |
| `slice_orthogonal()`     | X, Y, Z 軸に直交する3枚のスライスをまとめて取得 |


又、`Plotter` クラスの以下の3つのメソッドは、インタラクティブにスライス面を調整できるウィジェットを追加するためのものです。ユーザーは GUI 上でスライスの位置や形を自由に動かすことができ、非常に便利です。

| メソッド名                         | 概要                                           | ウィジェットの種類           |
|------------------------------------|------------------------------------------------|-------------------------------|
| `add_mesh_slice()`                | 任意の平面で1枚スライス                         | 平面ドラッグ                  |
| `add_mesh_slice_orthogonal()`     | X/Y/Z 軸方向に直交する3枚スライス               | 3軸スライダー                 |
| `add_mesh_slice_spline()`         | スプライン（曲線）に沿ったスライス              | スプラインの制御点移動可能    |


わかりやすくするために、次のコードではグリッドの外形枠を計算します。このコードは VTKライブラリのクラスを直接使用して外形枠を生成し、得られた結果を `pyvista.wrap()` を使って PyVista の `PolyData` に変換します。

In [4]:
sgof = vtk.vtkStructuredGridOutlineFilter()
sgof.SetInputData(grid)
sgof.Update()
outline = pv.wrap(sgof.GetOutput())

次のコードは、外形枠、`grid.slice_orthogonal()` によって得られる直交スライス、および自由にスライス平面を調整できるウィジェットを表示します。この表示にはインタラクティブなウィジェットが含まれているため、Jupyter Notebook 上では直接動作しません。そのため、`pyvistaqt.BackgroundPlotter()` を使って別ウィンドウで描画を行います。

![](images/pyvista-slice.png)

In [5]:
import pyvistaqt as pvqt

plotter = pvqt.BackgroundPlotter()
plotter.add_mesh(outline)
plotter.add_mesh(grid.slice_orthogonal(), cmap='cool')
plotter.add_mesh_slice(grid, cmap='cool')
plotter.show()

## 等値面

等値面とは、スカラー場において同じスカラー値を持つ点を結んでできる曲面のことです。地図における等高線に似ています。

PyVista では `contour` メソッドを使用して等値面を計算します。`isosurfaces` 引数には生成する等値面の数を、`scalars` 引数には使用するスカラー配列の名前を指定します。

以下のプログラムは、前節で紹介した流体データに対して等値面の可視化を行うものです。

In [6]:
plotter = pv.Plotter()
plotter.add_mesh(outline)
plotter.add_mesh(grid.contour(isosurfaces=8, scalars="Density"), cmap='cool', opacity=0.3)
plotter_to_iframe(plotter)

In [7]:
plotter = pvqt.BackgroundPlotter()
plotter.add_mesh(outline)
plotter.add_mesh_isovalue(grid, cmap='cool')
plotter.show()

![](images/pyvista-isovalue.png)

上の例では、同じ等値面上のすべての点の色は同じになっています。これは、等値面上のスカラー値（ここでは流体の密度）が同一であり、かつ等値面の着色にデフォルトでそのスカラー値が使われているためです。しかし、等値面の色を別のスカラー値に基づいて表示したい場合もあります。以下のプログラムは、異なるスカラー値を使って等値面に色を付ける方法を示しています。`contour()`の`isosurfaces`引数はリストの場合、そのリスト中の値に対応する面を作成します。`add_mesh()`の`scalars`引数で着色に使用する配列名を指定します。

In [8]:
plotter = pv.Plotter()
plotter.add_mesh(outline)
plotter.add_mesh(grid.contour(isosurfaces=[0.3], scalars="Density"), cmap='cool', scalars="VelocityMagnitude")
plotter_to_iframe(plotter)

In [9]:
plotter = pvqt.BackgroundPlotter()
plotter.add_mesh(outline)
plotter.add_mesh_isovalue(grid, cmap='cool')
plotter.show()

## ベクトルフィールド

前の例では、スライス面や等値面を用いて流体の密度分布を可視化しました。空間中の各点における密度は数値（スカラー）で表すことができるため、密度分布はスカラー場として理解できます。一方、流体の各点における速度はベクトルであるため、速度の分布はベクトル場で表現する必要があります。本節では、ランダムに散布されたベクトル矢印や流線を用いて、ベクトル場を可視化する方法について紹介します。

次のコードは、4つの要素で構成されているグラフを作成します：外枠、ランダムに散らばったベクトル矢印、流線の発生源を表す球体、そして流線です。

In [24]:
import vtk
mp = vtk.vtkMaskPoints() #{1}
mp.SetRandomMode(True)
mp.SetOnRatio(100)
mp.SetInputData(grid)
mp.Update()
rand_points = pv.wrap(mp.GetOutput())

sphere = pv.Sphere(radius=2, center=grid.center, phi_resolution=8, theta_resolution=8)
lines = grid.streamlines_from_source(sphere, integration_direction='forward')
tubes = lines.tube(radius=0.05)

plotter = pv.Plotter()
plotter.add_mesh(outline)
plotter.add_mesh(rand_points.glyph(factor=2.0), scalars='VelocityMagnitude')
plotter.add_mesh(sphere, style='wireframe')
plotter.add_mesh(tubes, scalars='VelocityMagnitude')
plotter_to_iframe(plotter)

In [52]:
import vtk

class StreamPlot(pvqt.BackgroundPlotter):
    def plot_stream(self, grid):
        self.grid = grid
        self.stream_actor = None
        
        mp = vtk.vtkMaskPoints() #{1}
        mp.SetRandomMode(True)
        mp.SetOnRatio(100)
        mp.SetInputData(grid)
        mp.Update()
        rand_points = pv.wrap(mp.GetOutput())

        sgof = vtk.vtkStructuredGridOutlineFilter()
        sgof.SetInputData(grid)
        sgof.Update()
        outline = pv.wrap(sgof.GetOutput())

        self.add_mesh(outline)
        # self.add_mesh(rand_points.glyph(factor=2.0), scalars='VelocityMagnitude')
        self.sphere_widget = self.add_sphere_widget(self.callback, center=grid.center, radius=2, theta_resolution=8, phi_resolution=8, style="wireframe", pass_widget=True)
        self.show()
    
    def callback(self, center, sphere_widget):
        sphere = pv.PolyData()
        sphere_widget.GetPolyData(sphere)
        lines = grid.streamlines_from_source(sphere, integration_direction='forward')
        tubes = lines.tube(radius=0.05)
        if self.stream_actor is None:
            self.stream_actor = self.add_mesh(tubes, scalars='VelocityMagnitude')
        else:
            arr = tubes['VelocityMagnitude']
            self.stream_actor.mapper.dataset = tubes
            self.stream_actor.mapper.scalar_range = arr.min(), arr.max()

plot = StreamPlot()
plot.plot_stream(grid)

## 円柱の交線を計算する

互いに垂直な2本の円筒が交差すると、8本の交線が生成されます。以下では、`PolyData`のブール演算機能を使って、互いに垂直な2本の円筒を生成し、それらの交線を計算します。

次の `make_tube()` 関数では、`Cylinder()` を使って円管の内径と外径を表す2つの円柱を作成し、`triangulate()` によってすべての面を三角形に変換します。さらに、`subdivide()` を用いて三角形を細かく分割します。`boolean_difference()` によって外径の円柱から内径の円柱を差し引いた形状（円管）を作成し、3つの `PolyData` を `Tube` クラスにまとめて返します。

なお、内径側の円柱を作成する際に高さを `+0.1` しているのは、`boolean_difference()` の処理を確実に成功させるための工夫です。

In [169]:
from dataclasses import dataclass

@dataclass
class Tube:
    inner: pv.PolyData
    outer: pv.PolyData
    tube: pv.PolyData

def make_tube(height, radius, resolution, direction):
    cs1 = pv.Cylinder(height=height, radius=radius[0], resolution=resolution, direction=direction)
    cs2 = pv.Cylinder(height=height+0.1, radius=radius[1], resolution=resolution, direction=direction)

    cs1 = cs1.triangulate().subdivide(nsub=1, subfilter='loop')
    cs2 = cs2.triangulate().subdivide(nsub=1, subfilter='loop')
    tube = cs1.boolean_difference(cs2)
    return Tube(outer=cs1, inner=cs2, tube=tube)

tube1 = make_tube(5, [1, 0.8], 32, (0, 0, 1))
tube2 = make_tube(5, [0.7, 0.55], 32, (0, 1, 0))

次に、2つの円管の形状をもとに、互いに干渉・重なり合う部分を処理します。`tube1` の円管から `tube2` の内径部分を差し引くことで、`tube2` が挿入された際に穴が空いた形状である `tube1_with_hole` を生成します。さらに、`boolean_intersection()` を用いて、`tube1` と `tube2` の円管の共通部分（重なり領域）を計算し、その交差形状を生成します。

In [170]:
tube1_with_hole = tube1.tube.boolean_difference(tube2.inner)
tube2_with_hole = tube2.tube.boolean_difference(tube1.inner)
intersection = tube1.tube.boolean_intersection(tube2.tube)

三つの部分をグラフにします。

In [171]:
plotter = pv.Plotter()
plotter.add_mesh(tube1_with_hole, color='gray', opacity=0.5)
plotter.add_mesh(tube2_with_hole, color='gray', opacity=0.5)
plotter.add_mesh(intersection, color='red')
plotter_to_iframe(plotter)

交線（共通部分のエッジ）を抽出するには、`extract_feature_edges()` を使用します。これは、メッシュ内のシャープなエッジを抽出する機能で、`feature_angle` 引数でエッジの鋭さのしきい値を角度で指定できます。この例では、デフォルト値の 30 度で交線を抽出できます。

`clean()` を使用すると、近接する点をマージできます。また、`lines_to_points` 引数を `False` に設定することで、退化した線が点に変換されるのを防げます。

さらに、`connectivity()` を用いることで、連結している部分を識別できます。この処理によって、`RegionId` という名前のスカラー値がデータに追加されます。

In [278]:
edges = intersection.extract_feature_edges().clean(tolerance=1e-2, lines_to_points=False).connectivity()
edges

PolyData (0x21ef5977820)
  N Cells:    2455
  N Points:   1237
  N Strips:   0
  X Bounds:   -6.966e-01, 6.966e-01
  Y Bounds:   -9.952e-01, 9.952e-01
  Z Bounds:   -6.966e-01, 6.966e-01
  N Arrays:   10

次に、`RegionId` の値に基づいて色分けで`edges`の表示を行います。

In [279]:
plotter = pv.Plotter()
plotter.add_mesh(edges, cmap='tab10', scalars='RegionId')
plotter_to_iframe(plotter)

`threshold()` は、指定した `scalars` に基づいて、値が指定範囲内にあるセルを抽出します。結果は `UnstructuredGrid` として返され、各エッジとそれに対応する点の関係は `cells` に格納されます。`N Cells` や `N Points` の数を確認すると、重複するエッジが含まれていることがわかります。


In [285]:
edge = edges.threshold((-0.1, 1.1), scalars='RegionId')
edge

UnstructuredGrid (0x21ef28de560)
  N Cells:    689
  N Points:   345
  X Bounds:   -6.966e-01, 6.966e-01
  Y Bounds:   -7.962e-01, 7.962e-01
  Z Bounds:   -6.966e-01, 6.966e-01
  N Arrays:   10

以下の `connect_lines()` 関数は、`cells` 属性の情報をもとに線分をつなぎ、曲線としてまとめるものです。`RegionId` 配列のユニークな値ごとにループ処理を行い、それぞれの `RegionId` に対応するセルを抽出します。そして、`connect_lines()` を使ってすべての接続された線分を抽出し、その中で点数が最も多い部分を選んで `pv.lines_from_points()` によって線に変換し、描画します。`connect_lines()` のアルゴリズムの詳細についてはここでは説明を省略しますので、興味のある読者はぜひ自分で解析してみてください。

In [283]:
from matplotlib.cm import tab10

def connect_lines(lines):
    edges = defaultdict(set)    
    for _, s, e in lines.reshape(-1, 3).tolist():
        edges[s].add(e)
        edges[e].add(s)

    while True:
        if not edges:
            break
        poly = [next(iter(edges.keys()))]
        while True:
            e = poly[-1]
            neighbours = edges[e]
            if not neighbours:
                break
            n = neighbours.pop()
            try:
                edges[n].remove(e)
            except:
                pass
            poly.append(n)
        yield poly
        edges = {k:v for k,v in edges.items() if v}

plotter = pv.Plotter()
for color, v in zip(tab10.colors, set(edges['RegionId'])):
    edge = edges.threshold((v - 0.1, v + 0.1), scalars='RegionId')
    loop = sorted(connect_lines(edge.cells), key=len)[-1]
    points = edge.points[loop]
    plotter.add_points(points, color='gray', opacity=0.5)
    plotter.add_mesh(pv.lines_from_points(points), line_width=2, color=color)
plotter_to_iframe(plotter)